In [ ]:
dbutils.widgets.text("ticker", "VOD")
dbutils.widgets.text("mic", "XLON")
dbutils.widgets.text("start_date", "2025-01-01")
dbutils.widgets.text("end_date", "2025-01-10")

ticker = dbutils.widgets.get("ticker")
mic = dbutils.widgets.get("mic")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

In [ ]:
# ── Table configuration ─────────────────────────────────────────────────
# Set USE_SAMPLE_DATA=true/false as an environment variable (or Databricks
# cluster/job env var) to switch between trial views (DD-1167) and full
# production views without changing any code.
# Override catalog names via MARKET_DATA_CATALOG / DERIVED_DATA_CATALOG.
import os as _os

_s = _os.environ.get("USE_SAMPLE_DATA", "").lower()
USE_SAMPLE_DATA = True if _s == "true" else (False if _s == "false" else False)

MARKET_DATA_CATALOG  = _os.environ.get("MARKET_DATA_CATALOG",  "market_data")
DERIVED_DATA_CATALOG = _os.environ.get("DERIVED_DATA_CATALOG", "derived_data")

if USE_SAMPLE_DATA:
    # DD-1167 trial views — 6-month rolling window, no subscription required
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.REF_GLOBAL_PROD"
    ANALYTICS_TBL               = f"{MARKET_DATA_CATALOG}.ANA_GLOBAL_PROD"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.TRDS_GLOBAL_PROD"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.TDSP_GLOBAL_PROD"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.L2_GLOBAL_PROD"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.CBBO_GLOBAL_PROD"
    DAILY_CLASSIFIED_TRADES_TBL = f"{MARKET_DATA_CATALOG}.DCT_GLOBAL_PROD"
else:
    # Full production views
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_reference"
    ANALYTICS_TBL               = f"{DERIVED_DATA_CATALOG}.vw_analytics"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.vw_trades"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.vw_trades_plus"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.vw_l2_data_equity"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.vw_cbbo_equity"
    DAILY_CLASSIFIED_TRADES_TBL = f"{DERIVED_DATA_CATALOG}.vw_daily_classified_trades"

## BMLL Notebook Educational Series
### Consolidating Best Bid/Offer Across Multiple Venues

### Why Consolidation Matters

**Multi-Venue Trading Reality**: Many securities trade on multiple exchanges simultaneously:
- **Primary exchange**: Original listing venue (highest volume, best prices typically)
- **Alternative venues**: MTFs (Multilateral Trading Facilities), dark pools, lit venues
- **SIP/Consolidated feeds**: Regulatory consolidated quotes (where available)

**The Consolidation Problem**: Best prices may differ across venues:
- Venue A: Bid $100.05, Ask $100.10
- Venue B: Bid $100.04, Ask $100.09
- Venue C: Bid $100.06, Ask $100.11
- **Consolidated Best (CBBO)**: Bid $100.06 (best), Ask $100.09 (best)

**Why It Matters for Analysis**:
- Traders execute against the **best available prices** (CBBO), not single-venue prices
- Regulatory requirements often reference consolidated quotes
- Market impact analysis needs to account for fragmented liquidity
- Order routing algorithms consider multi-venue depth

**This Notebook**: Demonstrates how to consolidate per-venue best bid/offer (BBO) quotes
into a single consolidated best bid/offer (CBBO) time series using BMLL data.

BMLL provides Level 1 (BBO) quotes from 100+ exchanges. This notebook shows how to:
1. Retrieve BBO for each venue where a security trades
2. Align quotes across venues to the same timestamp
3. Compute consolidated best bid (max bid across venues)
4. Compute consolidated best ask (min ask across venues)
5. Track which venue(s) have the best price

## Import Libraries and Setup

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns',None)

In [ ]:
# Expect an existing Spark session named `spark` in Databricks notebooks.

###
### Consolidated Volume

Instruments are used to group equivalent Listings together so that the can be easily compared. Fungible listings traded in the same currency are grouped in the same Instrument. For equities, this means grouping by fungibility, based on the primary market and currency. 

Note that an Instrument is not directly queryable in reference data - querying for an object_type equals Instrument will provide all Listings that have the same Instrument Id. However, certain analytics from the BMLL Data Feed are available using the instrument.

We utilise reference data fields such this InstrumentId or the ISIN to source consolidated volume using our classified trades time_series api.

In [ ]:
sql = f"""
SELECT * FROM {REFERENCE_TBL} WHERE `TICKER` = 'VOD' AND `MIC` = 'XLON'
"""
_spark_df = spark.sql(sql)
_spark_df.show()

In [ ]:
ct_sql = f"""SELECT * FROM {DAILY_CLASSIFIED_TRADES_TBL} WHERE `INSTRUMENT_ID` = 121429 AND `TRADE_DATE` BETWEEN '2025-01-01' AND '2025-01-10'"""
ct = spark.sql(ct_sql).toPandas()
ct

In [ ]:
isin_sql = f"""SELECT DISTINCT `INSTRUMENT_ID` FROM {REFERENCE_TBL} WHERE `ISIN` IN ('GB00BH4HKS39')"""
isin_iids = spark.sql(isin_sql).toPandas()["INSTRUMENT_ID"].tolist()
if isin_iids:
    iid_list = ", ".join(str(i) for i in isin_iids)
    ct_sql = f"""
        SELECT *
        FROM {DAILY_CLASSIFIED_TRADES_TBL}
        WHERE INSTRUMENT_ID IN ({iid_list})
          AND TRADE_DATE BETWEEN '2025-01-01' AND '2025-01-10'
    """
    ct = spark.sql(ct_sql).toPandas()
else:
    ct = pd.DataFrame()
ct

###
### Consolidated BBO: Precompted
We provide access to a precomputed 10 level consolidated BBO (either NBBO in the US or EBBO in Europe), with price levels, and aggregated sizes for a instrument across all listing venues.

In [ ]:
sql = f"""
SELECT * FROM {CBBO_TBL} WHERE `MIC` = 'XLON' AND `TICKER` = 'VOD' AND `TRADE_DATE` = '2024-11-01' ORDER BY `EVENT_TIMESTAMP` NULLS LAST LIMIT 5
"""
_spark_df = spark.sql(sql)
_spark_df.show()

###
### Consolidated BBO: On the fly
It is also possible by utilising our granular data to create your own consolidated book, with price levels, and aggregated sizes for a instrument across all listing venues.

## Building CBBO On-the-Fly

While BMLL provides precomputed 10-level consolidated books, we can also build custom consolidations
from granular per-venue BBO data. This gives flexibility to handle:
- Custom security universes
- Non-standard consolidation rules
- Historical periods before precomputed CBBO was available
- Specific use cases requiring different consolidation logic

### Helper Functions for CBBO Construction

## Main CBBO Consolidation Algorithm

### Algorithm Overview

The consolidation process:
1. **Fetch per-venue BBO**: Get best bid/ask from each venue separately
2. **Timestamp alignment**: Ensure all quotes are aligned to the same timeline
3. **Forward-fill gaps**: BBO doesn't update if there's no change; fill down last-seen price
4. **Compute consolidated prices**:
   - Best bid = Max(all bids) - highest price buyers willing to pay
   - Best ask = Min(all asks) - lowest price sellers willing to accept
5. **Aggregate sizes**: Sum sizes from all venues at the consolidated best prices

### Key Challenges Handled

- **Index alignment**: Venues have different quote timing; use forward-fill to align
- **Duplicate timestamps**: Some timestamps may repeat; use last-observation-carry-forward
- **Stale data**: When no new quotes, price remains valid until next update
- **Multiple venues at best**: Sum sizes when multiple venues share the best bid/ask

In [ ]:
def merge_duplicate_mics(bbos, mics):
    if len(mics) == len(set(mics)):
        return bbos, mics
    new_bbos = []
    seen = set()
    for mic in set(mics):
        indices = [i for i, m in enumerate(mics) if m == mic]
        merged = pd.concat([bbos[i] for i in indices])
        new_bbos.append(merged)
        seen.add(mic)
    return new_bbos, list(seen)

In [ ]:
def get_cbbo(iid, date, return_all=True):
    ref_sql = f"""
        SELECT LISTING_ID, MIC
        FROM {REFERENCE_TBL}
        WHERE INSTRUMENT_ID = {iid}
          AND REFERENCE_DATE = '{date}'
    """
    ref_df = spark.sql(ref_sql).toPandas()
    if ref_df.empty:
        return pd.DataFrame()

    listing_ids = ref_df["LISTING_ID"].tolist()
    lid_to_mic = dict(zip(ref_df["LISTING_ID"], ref_df["MIC"]))
    lid_list = ", ".join(str(l) for l in listing_ids)

    l2_sql = f"""
        SELECT
            BMLL_OBJECT_ID AS LISTING_ID,
            EVENT_TIMESTAMP,
            BID_PRICE1,
            BID_QUANTITY1,
            ASK_PRICE1,
            ASK_QUANTITY1
        FROM `market_data`.`vw_l2_data_equity`
        WHERE BMLL_OBJECT_ID IN ({lid_list})
          AND TRADE_DATE = '{date}'
        ORDER BY EVENT_TIMESTAMP
    """
    l2 = spark.sql(l2_sql).toPandas()
    if l2.empty:
        return pd.DataFrame()

    bbos = []
    mics = []
    for lid, group in l2.groupby("LISTING_ID"):
        mic = lid_to_mic.get(lid, f"LID_{lid}")
        df = group.rename(columns={
            "EVENT_TIMESTAMP": "event_timestamp",
            "BID_PRICE1": "best_bid_price",
            "BID_QUANTITY1": "best_bid_size",
            "ASK_PRICE1": "best_ask_price",
            "ASK_QUANTITY1": "best_ask_size",
        })[["event_timestamp", "best_bid_price", "best_ask_price", "best_bid_size", "best_ask_size"]]
        df = df.set_index("event_timestamp").rename(columns={
            "best_bid_price": f"best_bid_price_{mic}",
            "best_bid_size": f"best_bid_size_{mic}",
            "best_ask_price": f"best_ask_price_{mic}",
            "best_ask_size": f"best_ask_size_{mic}",
        })
        bbos.append(df.replace(np.nan, 0))
        mics.append(mic)

    bbos, mics = merge_duplicate_mics(bbos, mics)
    try:
        bbo_df = pd.concat(bbos, axis=1, sort=False)
    except Exception:
        bbos = [df.loc[~df.index.duplicated(keep="last")] for df in bbos]
        bbo_df = pd.concat(bbos, axis=1, sort=False)

    best_cols = [col for col in bbo_df.columns if "best_" in col]
    if not bbo_df.empty:
        bbo_df[best_cols] = bbo_df[best_cols].ffill()
    bbo_df = bbo_df.replace(0, np.nan)

    for mic in mics:
        bbo_df[f"best_bid_size_{mic}"] = bbo_df[f"best_bid_size_{mic}"].fillna(0)
        bbo_df[f"best_ask_size_{mic}"] = bbo_df[f"best_ask_size_{mic}"].fillna(0)

    bbo_df["cbb_price"] = bbo_df.filter(like="best_bid_price").max(axis=1)
    bbo_df["cbo_price"] = bbo_df.filter(like="best_ask_price").min(axis=1)
    bbo_df["cbo_size"] = 0
    bbo_df["cbb_size"] = 0

    for mic in mics:
        bbo_df["cbo_size"] += (bbo_df[f"best_ask_price_{mic}"] == bbo_df["cbo_price"]) * bbo_df[f"best_ask_size_{mic}"]
        bbo_df["cbb_size"] += (bbo_df[f"best_bid_price_{mic}"] == bbo_df["cbb_price"]) * bbo_df[f"best_bid_size_{mic}"]

    bbo_df = bbo_df.reset_index().drop_duplicates(subset=bbo_df.columns.difference(['event_timestamp']), keep='first')
    return bbo_df if return_all else bbo_df[['event_timestamp', "cbb_size", "cbb_price", "cbo_price", "cbo_size"]]

In [ ]:
iid = 121429
ref = spark.sql(f"SELECT * FROM market_data.vw_reference WHERE INSTRUMENT_ID = {iid} AND REFERENCE_DATE = '2024-07-02'").toPandas()

BAH = get_cbbo(iid, '2024-07-02', return_all=False)
BAH

In [ ]:
get_cbbo(iid, '2024-07-02', return_all=True).head()